# 05 — Experiment Review

Tujuan: menghitung sendiri summary paired-fold dari **historical experiment yang sudah consumed**. Ini untuk memahami metric/gate, bukan untuk tuning atau membuat eksperimen baru.

Gunakan artifact historical-development saja; jangan arahkan notebook ke fresh-forward/protected outcomes.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

FOLD_METRICS = Path(r"CHANGE_ME")
CONTROL_NAME = "CONTROL_FINANCIAL_ERA"
CHALLENGER_NAME = "V2_PLUS_FINANCIAL"
FOLD_COL = "fold"
CANDIDATE_COL = "candidate"

In [ ]:
if not FOLD_METRICS.exists():
    raise FileNotFoundError("Set FOLD_METRICS ke historical fold_metrics CSV/parquet yang sudah consumed.")

m = pd.read_parquet(FOLD_METRICS) if FOLD_METRICS.suffix.lower() == ".parquet" else pd.read_csv(FOLD_METRICS)
print("shape:", m.shape)
display(m.head())
print("columns:", list(m.columns))

## 1. Pastikan candidate/fold yang mau dibandingkan benar

In [ ]:
if CANDIDATE_COL not in m or FOLD_COL not in m:
    raise KeyError("Adjust CANDIDATE_COL/FOLD_COL to match this artifact schema.")

print("candidates:", sorted(m[CANDIDATE_COL].dropna().astype(str).unique()))
print("folds:", sorted(m[FOLD_COL].dropna().astype(str).unique()))

## 2. Paired PR-AUC delta

Delta dihitung **pada fold yang sama**: challenger minus control.

In [ ]:
PR_COL = "pr_auc"
if PR_COL not in m:
    raise KeyError(f"{PR_COL!r} not found; set PR_COL to the artifact's PR-AUC field.")

pair = (
    m[m[CANDIDATE_COL].isin([CONTROL_NAME, CHALLENGER_NAME])]
    .pivot(index=FOLD_COL, columns=CANDIDATE_COL, values=PR_COL)
    .dropna(subset=[CONTROL_NAME, CHALLENGER_NAME])
)
pair["delta_pr_auc"] = pair[CHALLENGER_NAME] - pair[CONTROL_NAME]
display(pair)

print("median delta:", pair["delta_pr_auc"].median())
print("q25 delta   :", pair["delta_pr_auc"].quantile(0.25))
print("positive folds:", int((pair["delta_pr_auc"] > 0).sum()), "/", len(pair))

## 3. Guardrail metrics (kalau tersedia)

In [ ]:
for metric in ["roc_auc", "q5_q1"]:
    if metric not in m.columns:
        print(metric, "not present in this artifact")
        continue
    p = (
        m[m[CANDIDATE_COL].isin([CONTROL_NAME, CHALLENGER_NAME])]
        .pivot(index=FOLD_COL, columns=CANDIDATE_COL, values=metric)
        .dropna(subset=[CONTROL_NAME, CHALLENGER_NAME])
    )
    print("\n", metric)
    print("control median   :", p[CONTROL_NAME].median())
    print("challenger median:", p[CHALLENGER_NAME].median())

## Cara membaca

- Positive median saja belum cukup kalau lower-quartile paired delta jelek.
- Paired comparison menjaga tiap challenger dibandingkan pada fold/population yang sama.
- Guardrail mencegah satu metric kecil yang membaik menutupi deterioration yang lebih luas.
- Jangan mengganti feature, estimator, threshold, atau horizon setelah melihat hasil ini lalu menyebutnya eksperimen yang sama. Itu post-outcome tuning.